### RAG Pipelines - Data Ingestion to vector DB pipeline

In [26]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path


In [27]:
### Read all the pdf's inside the directory


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documemnts = loader.load()

            # Add source information to metadata
            for doc in documemnts:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documemnts)
            print(f" Loaded {len(documemnts)} pages")
        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")
    

Found 2 PDF files to process

Processing: attention.pdf
 Loaded 2 pages

Processing: Resume.pdf
 Loaded 2 pages

Total documents loaded: 4


In [28]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m145 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Resume.docx', 'source': '../data/pdf/attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content="Sobhie\n \nSaad\n \nSenior\n \nSoftware\n \nEngineer\n \nExperienced\n \nFull-Stack\n \nDeveloper\n \nwith\n \nover\n \n8\n \nyears\n \nof\n \nexpertise\n \nin\n \nbackend\n \ntechnologies.\n \nSkilled\n \nin\n \ndeveloping\n \nscalable\n \napplications,\n \noptimizing\n \nperformance,\n \nmigrating\n \nand\n \nintegrating\n \nsystems.\n \nCurrent\n \nAddress:\n \nFrankfurt,\n \nGermany\n \nMobile:\n \n+49\n \n1787291538\n \n \nEmail\n \nSobhie.saad1@gmail.com\n             \n \nSocial\n:\n \nLinkedIn\n \n|\n \nGitHub\n \n \nGender:\n \nMale\n \n \nEXPERIENCE\n \nCheck24,\n \nFrankfurt,\n \nGermany\n \n—\n \nSenior\n \nSoftware\n \nEngineer\n \n|\n \nFull-time\n \n07\n \n2022\n \n-\n \npresent\n \n \nDe

In [29]:
### Text splitting get into chunks
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks for beter RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadate: {split_docs[0].metadata}")

    return split_docs

In [30]:
chunks = split_documents(all_pdf_documents)

Split 4 documents into 14 chunks

Example chunk:
Content: Sobhie
 
Saad
 
Senior
 
Software
 
Engineer
 
Experienced
 
Full-Stack
 
Developer
 
with
 
over
 
8
 
years
 
of
 
expertise
 
in
 
backend
 
technologies.
 
Skilled
 
in
 
developing
 
scalable
 
a...
Metadate: {'producer': 'Skia/PDF m145 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Resume.docx', 'source': '../data/pdf/attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


### Embedding and Vector Store DB

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/sobhiesaad/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sobhiesaad/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class EmpeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
        model_name: HuggingFace moden name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embeggin dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not found")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


In [4]:
## Initialize the embedding manager

embedding_manager = EmpeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded sucessfully. Embeggin dimension: 384


## Vector Store

In [37]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = '../data/vector_store'):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persisten ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name= self.collection_name,
                metadata= {"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embiddings to the vector store

        Args:
            documents: List of Langchain documents
            ebeddings: Corresponding ebeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError('Number of docs must match the number of embeddings')

        print(f"adding {len(documents)} documents to vector store ...")

        # Prepare data for ChromaDB
        ids = []
        metadata = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid5().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)

            # Document content
            documents_text.append(doc.page_content)

            # Embidding
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids= ids,
                embeddings= embeddings_list,
                metadatas= metadata,
                documents= documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise




In [38]:
vectorStore = VectorStore()
vectorStore

Vector store initialized. Collection pdf_documents
Existing documents in collection: 0


In [ ]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
texts

### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts= texts)

### Store in the vector database
# Replace the loop inside VectorStore.add_documents
metadata = []

for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
    unique_value = f"{doc.metadata.get('source_file', 'unknown')}:{i}:{doc.page_content}"
    doc_id = str(uuid.uuid5(uuid.NAMESPACE_URL, unique_value))
    ids.append(doc_id)

    chunk_metadata = dict(doc.metadata)
    chunk_metadata['doc_index'] = i
    chunk_metadata['content_length'] = len(doc.page_content)
    metadata.append(chunk_metadata)

    documents_text.append(doc.page_content)
    embeddings_list.append(embedding.tolist())

Generating embeddings for 14 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.41it/s]

Generated embeddings with shape: (14, 384)
adding 14 documents to vector store ...


TypeError: uuid5() missing 2 required positional arguments: 'namespace' and 'name'